# ICL-пайплайн: базовые стратегии отбора few-shot (random / balanced)

Обработка `id` приведена к формату из файла «Запрос 09 06 26»:
`id = f"{session_id}___{idx}"`, эмбеддинги грузятся позиционно
(индекс массива = индекс строки) в `df_*_with_embeddings`.


In [1]:
from __future__ import annotations

import math
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path
import time
import csv
import re

import asyncio
import nest_asyncio
import random
from collections import deque
from dataclasses import dataclass, field
from typing import Any, Callable, Iterable, Sequence

from openai import AsyncOpenAI, APIConnectionError, APIStatusError, RateLimitError
from tqdm.asyncio import tqdm


In [2]:
import logging
import sys
from io import StringIO
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert LOG_DIR.exists(), f"Не удалось создать директорию логов: {LOG_DIR}"

print(f"RUN_ID   : {RUN_ID}")
print(f"LOG_DIR  : {LOG_DIR}")

RUN_ID   : 20260618_101023
LOG_DIR  : logs


# Гиперпараметры модели 

In [3]:
from dotenv import load_dotenv



load_dotenv("/home/tanya/Programming/Projects/temp/justai_chatbot/doc_2026-04-27_19-27-23.env")
BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL_NAME = "YandexGPT-5-Lite-8B-instruct"
MAX_CONCURRENCY = 16
TEMPERATURE = 0
MAX_TEXT_LEN    = 3000


In [4]:
print(BASE_URL)

http://89.223.99.136:8101/v1


# Гиперпараметры отбора примеров ICL 

In [5]:
# Сетка количества примеров 
T_GRID = [2, 4, 8, 16, 32]

# Значения lambda 
LAMBDA_GRID = [0.5, 0.7, 0.9]

# Топ-M кандидатов по значению сходства 
MMR_TOP_M = 30

# Подстратегии:  
RUN_MMR_CLASSWISE = True # - разделение на legal/illegal
RUN_MMR_FLAT      = True # - отсутствие разделения на legal/illegal 

# Включение / исключение словаря 
RUN_DICT_VARIANTS = [True, False]  # [True] только со словарём, [False] только без

SEED_VALUE = 42
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)


# Загрузка словаря 

In [6]:
def load_drug_terms(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[tuple[str, str | None]] = set()
    items: list[dict] = []
    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            term = (row.get("normalized_term") or "").strip()
            cat_raw = (row.get("category") or "").strip()
            if not term:
                continue
            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue
            category: str | None = cat_raw if cat_raw else None
            key = (term, category)
            if key in seen:
                continue
            seen.add(key)
            items.append({"term": term, "category": category})
    items.sort(key=lambda x: (x["term"], x["category"] or ""))
    return json.dumps(items, ensure_ascii=False)


DRUG_TERMS = load_drug_terms("illegal_terms_dictionary_edit.csv")
print(f"Загружено терминов в DRUG_TERMS: {len(json.loads(DRUG_TERMS))}")


Загружено терминов в DRUG_TERMS: 754


# Загрузка документов 

In [7]:
# ─────────────────────────────────────────────
# ЗАГРУЗКА ДОКУМЕНТОВ
# id-формат как в «Запрос 09 06 26»: id = f"{session_id}___{idx}",
# где idx — порядковый индекс строки в датасете.
# ─────────────────────────────────────────────

df_val  = pd.read_parquet("/home/tanya/Programming/Projects/temp/justai_chatbot/data/processed/val.parquet").reset_index(drop=True)
df_test = pd.read_parquet("/home/tanya/Programming/Projects/temp/justai_chatbot/data/processed/test.parquet").reset_index(drop=True)


def row_to_input_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":   f"{row['session_id']}___{idx}",
        "text": text[:MAX_TEXT_LEN],
    }


# Основная (тестовая) выборка для расчёта метрик
test_input_jsons = [row_to_input_json(row, idx) for idx, row in df_test.iterrows()]
# input_jsons оставлен для совместимости с остальными ячейками (= тестовая выборка)
input_jsons = test_input_jsons

print(f"Всего записей в test: {len(test_input_jsons)}")
print(f"Уникальных id: {len({i['id'] for i in test_input_jsons})}")
print("\nПример:")
print(json.dumps(test_input_jsons[0], ensure_ascii=False, indent=2))

assert len({i["id"] for i in test_input_jsons}) == len(test_input_jsons), (
    "id не уникален в test.parquet!"
)


Всего записей в test: 305
Уникальных id: 305

Пример:
{
  "id": "telegram-8412110593-adyoika-8412110593-NOn-52042546487-0880052307.91918c58-bfb1-1a6f-91d5-930278a7f694___0",
  "text": "Вопрос: /newNode_2;Тбилиси თბილისი\nОтвет: Выберите район \n\nაირჩიეთ რაიონი."
}


In [8]:
# Превью id тестовой выборки в новом формате session_id___idx
df_test_preview = df_test.copy()
df_test_preview["id"] = (
    df_test_preview["session_id"].astype(str) + "___" + df_test_preview.index.astype(str)
)
print(df_test_preview[["id"]].head())


                                                  id
0  telegram-8412110593-adyoika-8412110593-NOn-520...
1  telegram-183263525-massatela1_startapy_-183263...
2  telegram-0509107573-shop_2-0509107573-lTb-0664...
3  telegram-322189240-onlajn_zakazy-322189240-dJG...
4  telegram-322189240-onlajn_zakazy-322189240-dJG...


In [9]:
# ─────────────────────────────────────────────
# ЗАГРУЗКА TRAIN (id-формат как в «Запрос 09 06 26»)
# ─────────────────────────────────────────────

df_train = pd.read_parquet("train.parquet").reset_index(drop=True)


def row_to_train_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":    f"{row['session_id']}___{idx}",
        "text":  text[:MAX_TEXT_LEN],
        "label": int(row["from_illegal_account"]),  # 0 = legal, 1 = illegal
    }


examples_jsons = [row_to_train_json(row, idx) for idx, row in df_train.iterrows()]
examples_id    = [it["id"]    for it in examples_jsons]
examples_text  = [it["text"]  for it in examples_jsons]
examples_label = [it["label"] for it in examples_jsons]

print(f"Всего записей в train : {len(examples_id)}")
print(f"Уникальных id         : {len(set(examples_id))}")
print(f"Legal   (0)           : {examples_label.count(0)}")
print(f"Illegal (1)           : {examples_label.count(1)}")
assert len(set(examples_id)) == len(examples_id), "id не уникален в train.parquet!"

legal_indices   = [i for i, lbl in enumerate(examples_label) if lbl == 0]
illegal_indices = [i for i, lbl in enumerate(examples_label) if lbl == 1]


Всего записей в train : 748
Уникальных id         : 748
Legal   (0)           : 182
Illegal (1)           : 566


# Реализация MMR 

In [10]:
def l2_normalized(x):
        x = x.astype("float32")
        norms = np.linalg.norm(x, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return x / norms

In [11]:
from collections import deque
from dataclasses import dataclass, field
from typing import Callable, Iterable, Sequence
       
# ════════════════════════════════════════════════════════════════════════
# MMR — ядро алгоритма диверсификации
# ════════════════════════════════════════════════════════════════════════
def mmr_select(
    test_vec: np.ndarray,        # [d] — нормализованный вектор тестового документа
    cand_vecs: np.ndarray,       # [M, d] — нормализованные векторы кандидатов
    n_select: int,               # сколько примеров отобрать
    lambda_: float = 0.7,        # баланс релевантность/разнообразие
) -> tuple[list[int], np.ndarray]:
    """
    Шаги:
        a. начинаем с пустого списка;
        b. первый пример = самый похожий на тестовый документ;
        c. для каждого оставшегося кандидата считаем MMR-оценку;
        d. берём кандидата с максимальной MMR-оценкой;
        e. повторяем, пока не наберём n_select.
    """
    M = cand_vecs.shape[0]
    n_select = min(n_select, M)
    if n_select <= 0 or M == 0:
        return [], np.zeros(M, dtype=np.float32)

    # близость каждого кандидата к тестовому документу: 
    sim_to_test = cand_vecs @ test_vec

    # попарная близость кандидатов друг к другу: 
    cand_sim = cand_vecs @ cand_vecs.T

    selected: list[int] = []
    remaining = list(range(M))

    # (b) первый — максимально похожий на тест
    first = int(np.argmax(sim_to_test))
    selected.append(first)
    remaining.remove(first)

    # близости каждого кандидата к уже выбранным
    max_sim_to_sel = cand_sim[:, first].copy()

    # добираем остальные
    while len(selected) < n_select and remaining:
        rem = np.asarray(remaining)
        mmr = lambda_ * sim_to_test[rem] - (1.0 - lambda_) * max_sim_to_sel[rem]
        best_local = int(np.argmax(mmr))
        best = int(rem[best_local])

        selected.append(best)
        remaining.remove(best)
        # обновляем running-max: добавился новый выбранный сосед
        max_sim_to_sel = np.maximum(max_sim_to_sel, cand_sim[:, best])

    return selected, sim_to_test

# ════════════════════════════════════════════════════════════════════════
# УПОРЯДОЧИВАНИЕ ПРИМЕРОВ В ПРОМПТЕ
# ════════════════════════════════════════════════════════════════════════
def order_for_prompt(
    pool_indices: Sequence[int],   # индексы выбранных примеров в общем пуле train
    sims: Sequence[float],         # близость каждого выбранного к тесту
    labels: Sequence[int],         # метка каждого выбранного (0=legal, 1=illegal)
) -> list[int]:
    """
    Упорядочивает выбранные примеры для финального промпта по правилам:
        - по возможности ЧЕРЕДОВАТЬ метки классов;
        - наиболее ПОХОЖИЕ примеры — БЛИЖЕ к тестовому документу.
    """
    items = list(zip(pool_indices, sims, labels))

    # очереди по классам, отсортированные по убыванию близости (самый похожий — сверху)
    by_label: dict[int, deque] = {
        0: deque(sorted([it for it in items if it[2] == 0], key=lambda x: -x[1])),
        1: deque(sorted([it for it in items if it[2] == 1], key=lambda x: -x[1])),
    }

    def top_sim(lbl: int) -> float:
        return by_label[lbl][0][1] if by_label[lbl] else -np.inf

    # стартуем с класса, чей самый похожий пример ближе к тесту
    cur = 0 if top_sim(0) >= top_sim(1) else 1

    near_to_far: list[tuple] = []  # от теста (самый похожий) наружу
    while by_label[0] or by_label[1]:
        if by_label[cur]:
            near_to_far.append(by_label[cur].popleft())
            cur = 1 - cur
        elif by_label[1 - cur]:
            cur = 1 - cur  # текущий класс кончился — переключаемся
        else:
            break

    # в промпте: дальние первыми, ближний (самый похожий) — последним
    prompt_order = list(reversed(near_to_far))
    return [it[0] for it in prompt_order]

# ════════════════════════════════════════════════════════════════════════
# СЕЛЕКТОР ПРИМЕРОВ  
# ════════════════════════════════════════════════════════════════════════
@dataclass
class MMRSelector:
    """
        select_classwise — деление на legal/illegal, top-M в каждом классе,
                            N/2 + N/2, MMR отдельно внутри каждого класса;
        select_flat      — без деления на классы, top-M по всему пулу, MMR.
    """
    train_emb: np.ndarray
    train_labels: np.ndarray
    M: int = 30                          # сколько кандидатов оставляем по близости
    lambda_default: float = 0.7

    # индексы по классам (заполняются в __post_init__)
    legal_idx: np.ndarray = field(init=False)
    illegal_idx: np.ndarray = field(init=False)

    def __post_init__(self):
        self.train_emb = l2_normalized(self.train_emb)
        self.train_labels = np.asarray(self.train_labels).astype(int)
        self.legal_idx = np.where(self.train_labels == 0)[0]
        self.illegal_idx = np.where(self.train_labels == 1)[0]

    #top-M по близости + MMR внутри подмножества 
    def _topM_then_mmr(
        self,
        test_vec: np.ndarray,
        subset_idx: np.ndarray,   # индексы кандидатов в общем пуле
        n_select: int,
        lambda_: float,
    ) -> tuple[list[int], list[float]]:
        if n_select <= 0 or len(subset_idx) == 0:
            return [], []

        sub_emb = self.train_emb[subset_idx]    
        sim = sub_emb @ test_vec # близость к тесту

        # (b) оставляем только top-M кандидатов по близости.
        # Эффективный M не может быть меньше n_select, иначе не наберём нужное
        m = min(max(self.M, n_select), len(subset_idx))
        topm_local = np.argsort(-sim)[:m] # позиции внутри subset
        cand_idx = subset_idx[topm_local] # индексы в общем пуле
        cand_emb = self.train_emb[cand_idx]

        # MMR внутри отобранных кандидатов
        order_local, sim_to_test = mmr_select(
            test_vec, cand_emb, n_select=n_select, lambda_=lambda_
        )
        chosen_pool_idx = [int(cand_idx[i]) for i in order_local]
        chosen_sims = [float(sim_to_test[i]) for i in order_local]
        return chosen_pool_idx, chosen_sims

    # РЕЖИМ 1: деление на классы (balanced N/2 + N/2)
    def select_classwise(
        self,
        test_vec: np.ndarray,
        n_total: int,
        lambda_: float | None = None,
    ) -> list[int]:
        
        lam = self.lambda_default if lambda_ is None else lambda_

        n_illegal = n_total // 2
        n_legal = n_total - n_illegal  # при нечётном n_total legal получает +1

        # защита от нехватки в пуле
        n_legal = min(n_legal, len(self.legal_idx))
        n_illegal = min(n_illegal, len(self.illegal_idx))

        legal_pick, legal_sims = self._topM_then_mmr(
            test_vec, self.legal_idx, n_legal, lam
        )
        illegal_pick, illegal_sims = self._topM_then_mmr(
            test_vec, self.illegal_idx, n_illegal, lam
        )

        pool_idx = legal_pick + illegal_pick
        sims = legal_sims + illegal_sims
        labels = [0] * len(legal_pick) + [1] * len(illegal_pick)
        return order_for_prompt(pool_idx, sims, labels)

    # РЕЖИМ 2: без деления на классы
    def select_flat(
        self,
        test_vec: np.ndarray,
        n_total: int,
        lambda_: float | None = None,
    ) -> list[int]:

        lam = self.lambda_default if lambda_ is None else lambda_
        all_idx = np.arange(self.train_emb.shape[0])
        pool_idx, sims = self._topM_then_mmr(test_vec, all_idx, n_total, lam)
        labels = [int(self.train_labels[i]) for i in pool_idx]
        return order_for_prompt(pool_idx, sims, labels)

# Эмбеддинги и селектор

In [12]:
from pathlib import Path
import numpy as np

# ─────────────────────────────────────────────
# ЗАГРУЗКА ЭМБЕДДИНГОВ (логика как в «Запрос 09 06 26»)
# Убрана сложная система маппинга по *_ids.npy.
# Индекс в массиве эмбеддингов = порядковый индекс строки в датасете.
# Создаются объединённые датасеты df_*_with_embeddings.
# ─────────────────────────────────────────────

EMB_DIR = Path("/home/tanya/Programming/Projects/temp/justai_chatbot/embeddings_out_qwen3_8b/embeddings_out_qwen3_8b")


def load_embeddings_by_index(split_name: str, emb_dir: Path = EMB_DIR) -> np.ndarray:
    """Загружает эмбеддинги сплита; индекс массива соответствует индексу строки датасета."""
    emb_path = emb_dir / f"{split_name}_embeddings.npy"
    if not emb_path.exists():
        raise FileNotFoundError(f"Файл эмбеддингов не найден: {emb_path}")

    embeddings = np.load(emb_path, allow_pickle=True).astype(np.float32)

    n_check = min(100, len(embeddings))
    sample_norms = np.linalg.norm(embeddings[:n_check], axis=1)
    already_normalized = bool(np.allclose(sample_norms, 1.0, atol=1e-3))
    if not already_normalized:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        embeddings = (embeddings / norms).astype(np.float32)

    print(f"  [{split_name}] shape={embeddings.shape} | normalized={already_normalized}")
    return embeddings


def _attach_embeddings(df, split_name):
    """Возвращает (df_with_emb, ok). При отсутствии/несовпадении эмбеддингов ok=False."""
    try:
        emb = load_embeddings_by_index(split_name)
    except FileNotFoundError as e:
        print(f"  [{split_name}] ⚠ {e} — пропускаю (нужно только для семантических стратегий)")
        return None, False
    if len(df) != len(emb):
        print(f"  [{split_name}] ⚠ размер датасета ({len(df)}) != эмбеддингов ({len(emb)}) — пропускаю")
        return None, False
    out = df.copy()
    out["embedding"] = list(emb)
    return out, True


print("\nЗагружаем эмбеддинги...")
df_train_with_embeddings, _train_ok = _attach_embeddings(df_train, "train")
df_val_with_embeddings,   _val_ok   = _attach_embeddings(df_val,   "val")
df_test_with_embeddings,  _test_ok  = _attach_embeddings(df_test,  "test")

# ─────────────────────────────────────────────
# Селектор MMR (используется только динамическими стратегиями)
# ─────────────────────────────────────────────
mmr_selector = None
if _train_ok:
    train_emb_matrix = np.stack(df_train_with_embeddings["embedding"].values)
    mmr_selector = MMRSelector(
        train_emb=train_emb_matrix,
        train_labels=np.array(examples_label),
        M=MMR_TOP_M,
        lambda_default=LAMBDA_GRID[0],
    )
    print(f"\n✅ MMRSelector построен из df_train_with_embeddings ({len(df_train_with_embeddings)} записей)")
else:
    print("\n⚠ MMRSelector не построен (нет эмбеддингов train). Базовым стратегиям он не нужен.")

# Датасет, на котором считаем метрики (тест) — отсюда берутся query-эмбеддинги
INPUT_DF_WITH_EMB = df_test_with_embeddings


def get_test_vec(item):
    """
    Достаёт query-эмбеддинг по item['id'] = 'session_id___idx',
    как в build_messages_semantic из «Запрос 09 06 26»: row_idx → .iloc[row_idx].
    """
    if INPUT_DF_WITH_EMB is None:
        raise KeyError("Эмбеддинги тестовой выборки не загружены")
    parts = str(item["id"]).split("___")
    if len(parts) != 2:
        raise KeyError(f"Не удалось разобрать id={item['id']} в формате session_id___idx")
    row_idx = int(parts[1])
    return INPUT_DF_WITH_EMB.iloc[row_idx]["embedding"]


def make_mmr_selector_fn(classwise: bool, lam: float):
    def _fn(t, item):
        tv = get_test_vec(item)
        if classwise:
            return mmr_selector.select_classwise(tv, t, lambda_=lam)
        return mmr_selector.select_flat(tv, t, lambda_=lam)
    return _fn



Загружаем эмбеддинги...
  [train] shape=(748, 4096) | normalized=False
  [val] shape=(336, 4096) | normalized=False
  [test] shape=(305, 4096) | normalized=False

✅ MMRSelector построен из df_train_with_embeddings (748 записей)


# ПРОМПТЫ

In [13]:
SYSTEM_PROMPT = "Ты - помощник по классификации текста для задачи модерации на предмет упоминания наркотиков."
prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ:
{DRUG_TERMS}

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕРЫ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ:
{FEW_SHOT_EXAMPLES}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""
prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Прямое упоминание сущностей, связанных с наркотиками и наркоторговлей:
   - обменник, клад, кладмен, фасовка, закладка и т.д.

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова, ососбенно в названиях каналов и ботов через @:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - добавление лишних символов: DeaIler
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Подозрительные аббревиатуры и одиночные буквы латиницей в качестве вопроса:
   - Bbgg, Sh, I, CV GK j

6. Упоминания криптокошельков и криптовалют

7. Фразы с двойным дном и иносказания:
   -  Главное не забывать: счастье — это когда ты нашёл, а тебя нет!

8. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

9. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ПРИМЕРЫ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ:
{FEW_SHOT_EXAMPLES}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

In [14]:
KNOWN_PLACEHOLDERS = {"FEW_SHOT_EXAMPLES", "DRUG_TERMS", "INPUT_JSON"}

def escape_unknown_placeholders(text: str, known: set) -> str:
    def replacer(m):
        key = m.group(1)
        return f"{{{{{key}}}}}" if key not in known else m.group(0)
    return re.sub(r'\{(\w+)\}', replacer, text)

prompt_d = escape_unknown_placeholders(prompt_d, KNOWN_PLACEHOLDERS)
print("Оставшиеся плейсхолдеры:", re.findall(r'\{(\w+)\}', prompt_d))


Оставшиеся плейсхолдеры: ['DRUG_TERMS', 'FEW_SHOT_EXAMPLES', 'INPUT_JSON']


In [15]:
# ─────────────────────────────────────────────
# Построитель few-shot блока.
# preserve_order=True → НЕ сортируем индексы.
# ─────────────────────────────────────────────
def build_few_shot_block(indices: list, preserve_order: bool = True) -> str:
    seq = list(indices) if preserve_order else sorted(indices)
    blocks = []
    for idx in seq:
        label = bool(examples_label[idx] == 1)
        label_str = "ILLEGAL" if label else "LEGAL"
        example_input  = json.dumps({"text": examples_text[idx]}, ensure_ascii=False, indent=2)
        example_output = json.dumps({"has_drug_mention": label}, ensure_ascii=False, indent=2)
        blocks.append(
            f"[{label_str}]\n"
            f"ПРИМЕР ВХОДА:\n{example_input}\n\n"
            f"ПРИМЕР ВЫХОДА:\n{example_output}"
        )
    return "\n\n" + ("\n\n" + "─" * 40 + "\n\n").join(blocks) + "\n"


# ─────────────────────────────────────────────
# Построитель сообщений.
# sample_fn имеет сигнатуру (t, item).
# ─────────────────────────────────────────────
def build_messages_d(item, sample_fn=None, use_dict: bool = True, t: int = 10):
    if sample_fn is not None:
        indices = sample_fn(t, item)
        few_shot_block = build_few_shot_block(indices, preserve_order=True)
    else:
        few_shot_block = "(примеры не используются)"

    drug_terms_value = DRUG_TERMS if use_dict else "(словарь не используется)"
    user_content = prompt_d.format(
        FEW_SHOT_EXAMPLES=few_shot_block,
        DRUG_TERMS=drug_terms_value,
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


# ─────────────────────────────────────────────
# БАЗОВЫЕ СТРАТЕГИИ ОТБОРА ПРИМЕРОВ (как в «Запрос 09 06 26»)
# ─────────────────────────────────────────────
def sample_random(t: int) -> list:
    """Случайная выборка t примеров."""
    return random.sample(range(len(examples_id)), t)


def sample_balanced(t: int) -> list:
    """Сбалансированная выборка t примеров (1:1 legal:illegal)."""
    n_illegal = t // 2
    n_legal   = t // 2
    remainder = t % 2

    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)

    if remainder:
        extra_pool = (
            [i for i in legal_indices   if i not in sampled_legal] +
            [i for i in illegal_indices if i not in sampled_illegal]
        )
        sampled_illegal += random.sample(extra_pool, remainder)

    result = sampled_legal + sampled_illegal
    assert len(result) == t, f"balanced: ожидалось {t}, получено {len(result)}"
    return result


def _wrap_static(fn):
    return lambda t, item: fn(t)  # статический семплер игнорирует item


# ─────────────────────────────────────────────
# ГЕНЕРАЦИЯ ВАРИАНТОВ ПРОМПТОВ
# Имя: prompt_d_{strategy}_{dict}_{t}exmpls[_lamXX]
# ─────────────────────────────────────────────
def _wrap_static(fn):
    return lambda t, item: fn(t)        # статический семплер игнорирует item

def make_variant_name(strategy, use_dict, t, lam=None):
    d = "with_dict" if use_dict else "wo_dict"
    lam_s = f"_lam{int(round(lam*100)):02d}" if lam is not None else ""
    return f"prompt_d_{strategy}_{d}_{t}exmpls{lam_s}"

PROMPT_VARIANTS: dict[str, Callable] = {}

# 2) Динамические MMR-стратегии (по сетке lambda)
_DYN = []
if RUN_MMR_CLASSWISE: _DYN.append(("mmr_classwise", True))
if RUN_MMR_FLAT:      _DYN.append(("mmr_flat",      False))

for _strat, _classwise in _DYN:
    for _lam in LAMBDA_GRID:
        _fn = make_mmr_selector_fn(_classwise, _lam)
        for _ud in RUN_DICT_VARIANTS:
            for _t in T_GRID:
                _name = make_variant_name(_strat, _ud, _t, _lam)
                PROMPT_VARIANTS[_name] = (
                    lambda x, f=_fn, u=_ud, tv=_t:
                        build_messages_d(x, sample_fn=f, use_dict=u, t=tv))

print(f"Всего вариантов промптов: {len(PROMPT_VARIANTS)}")
for _name in PROMPT_VARIANTS:
    print(" ", _name)


Всего вариантов промптов: 60
  prompt_d_mmr_classwise_with_dict_2exmpls_lam50
  prompt_d_mmr_classwise_with_dict_4exmpls_lam50
  prompt_d_mmr_classwise_with_dict_8exmpls_lam50
  prompt_d_mmr_classwise_with_dict_16exmpls_lam50
  prompt_d_mmr_classwise_with_dict_32exmpls_lam50
  prompt_d_mmr_classwise_wo_dict_2exmpls_lam50
  prompt_d_mmr_classwise_wo_dict_4exmpls_lam50
  prompt_d_mmr_classwise_wo_dict_8exmpls_lam50
  prompt_d_mmr_classwise_wo_dict_16exmpls_lam50
  prompt_d_mmr_classwise_wo_dict_32exmpls_lam50
  prompt_d_mmr_classwise_with_dict_2exmpls_lam70
  prompt_d_mmr_classwise_with_dict_4exmpls_lam70
  prompt_d_mmr_classwise_with_dict_8exmpls_lam70
  prompt_d_mmr_classwise_with_dict_16exmpls_lam70
  prompt_d_mmr_classwise_with_dict_32exmpls_lam70
  prompt_d_mmr_classwise_wo_dict_2exmpls_lam70
  prompt_d_mmr_classwise_wo_dict_4exmpls_lam70
  prompt_d_mmr_classwise_wo_dict_8exmpls_lam70
  prompt_d_mmr_classwise_wo_dict_16exmpls_lam70
  prompt_d_mmr_classwise_wo_dict_32exmpls_lam70
  p

In [16]:
class ColaFormatter(logging.Formatter):
    COLORS = {logging.DEBUG:"\033[37m", logging.INFO:"\033[36m",
              logging.WARNING:"\033[33m", logging.ERROR:"\033[31m",
              logging.CRITICAL:"\033[35m"}
    RESET = "\033[0m"
    def format(self, record):
        color = self.COLORS.get(record.levelno, self.RESET)
        record.levelname = f"{color}{record.levelname:<8}{self.RESET}"
        return super().format(record)

def setup_logger(name, log_file, console_level=logging.INFO, file_level=logging.DEBUG):
    logger = logging.getLogger(name); logger.setLevel(logging.DEBUG)
    if logger.handlers: logger.handlers.clear()
    file_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    console_fmt = ColaFormatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S")
    fh = logging.FileHandler(LOG_DIR / log_file, encoding="utf-8", mode="a")
    fh.setLevel(file_level); fh.setFormatter(file_fmt); logger.addHandler(fh)
    ch = logging.StreamHandler(sys.stdout); ch.setLevel(console_level); ch.setFormatter(console_fmt); logger.addHandler(ch)
    buffer = StringIO(); bh = logging.StreamHandler(buffer); bh.setLevel(logging.DEBUG); bh.setFormatter(file_fmt); logger.addHandler(bh)
    logger.buffer = buffer
    return logger

logger       = setup_logger("main",  f"run_{RUN_ID}.log",          console_level=logging.INFO)
api_logger   = setup_logger("api",   f"api_{RUN_ID}.log",          console_level=logging.WARNING)
parse_logger = setup_logger("parse", f"parse_errors_{RUN_ID}.log", console_level=logging.WARNING)
logger.info(f"Логирование настроено | RUN_ID={RUN_ID}")


10:10:23 | INFO     | main | Логирование настроено | RUN_ID=20260618_101023


# АСИНХРОННЫЕ ЗАПРОСЫ

In [17]:
async def send_one_request(client, model_name, messages, item_id="unknown"):
    last_error = None
    for attempt in range(5):
        start = time.time()
        try:
            response = await client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_tokens=64,
                temperature=0.0,
                seed=42,
                timeout=180,
            )
            elapsed = time.time() - start
            content = response.choices[0].message.content
            pt = response.usage.prompt_tokens
            ct = response.usage.completion_tokens
            api_logger.info(f"[{item_id}] tokens: prompt={pt} completion={ct} total={pt+ct}")
            return {"response": content, "time": elapsed,
                    "prompt_tokens": pt, "completion_tokens": ct, "total_tokens": pt+ct}
        except Exception as e:
            last_error = e
            api_logger.warning(f"[{item_id}] attempt={attempt+1}/5 {type(e).__name__}: {str(e)[:200]}")
            await asyncio.sleep(min(2 ** attempt, 20))
    api_logger.error(f"[{item_id}] failed after retries: {last_error}")
    raise last_error

async def process_with_semaphore(client, model_name, messages, item_id="unknown"):
    async with semaphore:
        return await send_one_request(client, model_name, messages, item_id=item_id)

async def safe_process(client, model_name, messages, item_id="unknown"):
    """Не даёт одной ошибке уронить весь gather — возвращает исключение как значение."""
    try:
        return await process_with_semaphore(client, model_name, messages, item_id=item_id)
    except Exception as e:
        logger.exception(f"Request failed for {item_id}: {e}")
        return e

def parse_response(raw: str, item_id: str):
    try:
        cleaned = re.sub(r"```(?:json)?|```", "", str(raw), flags=re.I).strip()

        # Ищем первый JSON-объект даже если модель добавила текст.
        match = re.search(r'\{.*\}', cleaned, flags=re.S)
        if match:
            cleaned = match.group(0)

        data = json.loads(cleaned)

        value = data.get("has_drug_mention")
        if isinstance(value, str):
            value = value.strip().lower() in {"true", "1", "yes"}

        return {
            "id": str(data.get("id") or item_id),
            "has_drug_mention": bool(value),
        }
    except Exception as e:
        parse_logger.error(f"[{item_id}] ParseError: {type(e).__name__}: {e} | raw={str(raw)[:500]!r}")
        return None


# ОЦЕНКА МЕТРИК

In [18]:
def evaluate_results(results_file, df_truth, label, truth_label_col="message_label"):
    if not Path(results_file).exists():
        print(f"[{label}] файл {results_file} не найден"); return None
    with open(results_file) as f:
        preds = json.load(f)
    df_pred = pd.DataFrame(preds)
    if df_pred.empty:
        print(f"[{label}] файл пустой"); return None
    df_pred = df_pred.drop_duplicates(subset="id", keep="last")
    df_pred["id"] = df_pred["id"].astype(str)

    # composite_id как в «Запрос 09 06 26»: session_id___index
    df_truth_local = df_truth.copy()
    df_truth_local["composite_id"] = (
        df_truth_local["session_id"].astype(str) + "___" +
        df_truth_local.index.astype(str)
    )

    expected_ids = set(df_truth_local["composite_id"])
    actual_ids   = set(df_pred["id"])
    missing = expected_ids - actual_ids
    extra   = actual_ids   - expected_ids
    if missing or extra:
        print(f"[{label}] ВНИМАНИЕ: пропущено id из truth: {len(missing)}, лишних id в pred: {len(extra)}")

    df_merged = df_truth_local.merge(
        df_pred, left_on="composite_id", right_on="id", how="inner"
    )
    if len(df_merged) == 0:
        print(f"[{label}] нет совпадений по id"); return None
    if len(df_merged) != len(df_truth_local):
        print(f"[{label}] предупреждение: смержилось {len(df_merged)} из {len(df_truth_local)} строк")

    y_true = (df_merged[truth_label_col] == "illegal").astype(int)
    y_pred = df_merged["has_drug_mention"].astype(int)
    metrics = {
        "version": label, "n": len(df_merged),
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1_binary": f1_score(y_true, y_pred, average="binary", zero_division=0),
        "f1_macro":  f1_score(y_true, y_pred, average="macro",  zero_division=0),
    }
    print(f"\n=== {label} (n={metrics['n']}) === f1_macro={metrics['f1_macro']:.4f} "
          f"f1_binary={metrics['f1_binary']:.4f} acc={metrics['accuracy']:.4f}")
    return metrics


# ГЛАВНАЯ ФУНКЦИЯ

In [19]:
async def run_variant(client, variant_name, items):
    """Классифицирует всю выборку одним вариантом промпта."""
    logger.info(f"{'='*50}")
    logger.info(f"Вариант: {variant_name} | записей: {len(items)}")

    builder = PROMPT_VARIANTS[variant_name]
    output_file = f"results_{variant_name}.json"

    all_parsed = []
    t0 = time.time()

    # Создаем задачи для всей выборки
    tasks = [
        asyncio.create_task(
            safe_process(
                client,
                MODEL_NAME,
                builder(item),
                item_id=item["id"]
            )
        )
        for item in items
    ]

    raw = await asyncio.gather(
    *tasks,
    return_exceptions=True
)

    valid = [r for r in raw if not isinstance(r, Exception)]

    for item, r in zip(items, raw):
        if isinstance(r, Exception):
            logger.warning(f"⚠️ ошибка id={item['id']}: {r}")
            continue

        res = parse_response(r["response"], item["id"])
        if res:
            all_parsed.append(res)

    # Сохраняем результаты
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_parsed, f, ensure_ascii=False, indent=2)

    total_time = time.time() - t0

    stats = None
    if valid:
        sp = sum(r["prompt_tokens"] for r in valid)
        sc = sum(r["completion_tokens"] for r in valid)
        st = sum(r["total_tokens"] for r in valid)
        n = len(valid)

        logger.info(
            f"[{variant_name}] ТОКЕНЫ | "
            f"ок={n}/{len(items)} | "
            f"prompt(sum={sp}, avg={sp/n:.1f}) "
            f"completion(sum={sc}, avg={sc/n:.1f}) "
            f"total(sum={st}, avg={st/n:.1f}) | "
            f"time={total_time:.1f}s"
        )

        stats = {
            "time": {
                "total": total_time,
                "avg": sum(r["time"] for r in valid) / n,
            },
            "prompt": {
                "sum": sp,
                "avg": sp / n,
            },
            "completion": {
                "sum": sc,
                "avg": sc / n,
            },
            "total": {
                "sum": st,
                "avg": st / n,
            },
        }

    logger.info(f"Сохранено {len(all_parsed)} результатов → {output_file}")

    return all_parsed, stats


async def main():
    global semaphore

    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    import httpx
    from openai import AsyncOpenAI

    http_client = httpx.AsyncClient(
        limits=httpx.Limits(
        max_connections=16,
        max_keepalive_connections=8,
    ),
        timeout=180,
    )

    print("BASE_URL =", BASE_URL)
    print("API_KEY loaded =", bool(API_KEY))

    client = AsyncOpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
        http_client=http_client,
    )

    # Классификация и метрики — на ТЕСТОВОЙ выборке
    items = test_input_jsons

    logger.info(
        f"Модель: {MODEL_NAME} | "
        f"вариантов: {len(PROMPT_VARIANTS)} | "
        f"записей (test): {len(items)}"
    )

    # ─────────────────────────────────────────────────────
    # Шаг 1. Инференс на тестовой выборке
    # ─────────────────────────────────────────────────────
    all_stats = {}

    for variant_name in PROMPT_VARIANTS:
        _, stats = await run_variant(
            client,
            variant_name,
            items,
        )

        if stats:
            all_stats[variant_name] = stats

    # ─────────────────────────────────────────────────────
    # Шаг 2. Метрики на ТЕСТЕ
    # ─────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("МЕТРИКИ НА ТЕСТОВОЙ ВЫБОРКЕ (test.parquet)")
    print(f"{'='*60}")

    all_metrics = []

    for variant_name in PROMPT_VARIANTS:
        m = evaluate_results(
            f"results_{variant_name}.json",
            df_test,
            variant_name,
            "message_label",
        )

        if m:
            all_metrics.append(m)

    if all_metrics:
        df_metrics = (
            pd.DataFrame(all_metrics)
            .set_index("version")
            .round(4)
            .sort_values("f1_macro", ascending=False)
        )

        print("\n=== Сводная таблица (test) ===")
        print(df_metrics)

        df_metrics.to_csv(f"metrics_test_{RUN_ID}.csv")

        best = df_metrics["f1_macro"].idxmax()

        print(
            f"\n🏆 Лучший по f1_macro: "
            f"{best} ({df_metrics.loc[best, 'f1_macro']:.4f})"
        )

    # ─────────────────────────────────────────────────────
    # Шаг 3. Токены и время
    # ─────────────────────────────────────────────────────
    if all_stats:
        rows = [
            {
                "variant": v,
                "time_total_s": round(s["time"]["total"], 1),
                "prompt_avg": round(s["prompt"]["avg"], 1),
                "completion_avg": round(s["completion"]["avg"], 1),
                "total_avg": round(s["total"]["avg"], 1),
                "total_sum": s["total"]["sum"],
            }
            for v, s in all_stats.items()
        ]

        df_time = pd.DataFrame(rows).set_index("variant")

        print(f"\n=== ТОКЕНЫ И ВРЕМЯ (test) ===")
        print(df_time)

        df_time.to_csv(f"time_tokens_test_{RUN_ID}.csv")


if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())


BASE_URL = http://89.223.99.136:8101/v1
API_KEY loaded = True
10:10:23 | INFO     | main | Модель: YandexGPT-5-Lite-8B-instruct | вариантов: 60 | записей (test): 305
10:10:23 | INFO     | main | ==================================================
10:10:23 | INFO     | main | Вариант: prompt_d_mmr_classwise_with_dict_2exmpls_lam50 | записей: 305
10:10:42 | INFO     | main | [prompt_d_mmr_classwise_with_dict_2exmpls_lam50] ТОКЕНЫ | ок=305/305 | prompt(sum=3404182, avg=11161.3) completion(sum=5091, avg=16.7) total(sum=3409273, avg=11177.9) | time=18.9s
10:10:42 | INFO     | main | Сохранено 305 результатов → results_prompt_d_mmr_classwise_with_dict_2exmpls_lam50.json
10:10:42 | INFO     | main | ==================================================
10:10:42 | INFO     | main | Вариант: prompt_d_mmr_classwise_with_dict_4exmpls_lam50 | записей: 305
10:11:03 | INFO     | main | [prompt_d_mmr_classwise_with_dict_4exmpls_lam50] ТОКЕНЫ | ок=305/305 | prompt(sum=3477150, avg=11400.5) completion(sum=

In [20]:
# Повторный расчёт метрик на ТЕСТЕ по сохранённым results_*.json
all_metrics = []

for variant_name in PROMPT_VARIANTS:
    m = evaluate_results(
        f"results_{variant_name}.json",
        df_test,
        variant_name,
        "message_label",
    )
    if m:
        all_metrics.append(m)

if all_metrics:
    df_metrics = (
        pd.DataFrame(all_metrics)
        .set_index("version")
        .round(4)
        .sort_values("f1_macro", ascending=False)
    )
    print("\n=== Сводная таблица (test) ===")
    print(df_metrics)



=== prompt_d_mmr_classwise_with_dict_2exmpls_lam50 (n=305) === f1_macro=0.6393 f1_binary=0.6382 acc=0.6393

=== prompt_d_mmr_classwise_with_dict_4exmpls_lam50 (n=305) === f1_macro=0.6588 f1_binary=0.6510 acc=0.6590

=== prompt_d_mmr_classwise_with_dict_8exmpls_lam50 (n=305) === f1_macro=0.4941 f1_binary=0.5787 acc=0.5082

=== prompt_d_mmr_classwise_with_dict_16exmpls_lam50 (n=305) === f1_macro=0.4741 f1_binary=0.5706 acc=0.4918

=== prompt_d_mmr_classwise_with_dict_32exmpls_lam50 (n=305) === f1_macro=0.4941 f1_binary=0.5787 acc=0.5082

=== prompt_d_mmr_classwise_wo_dict_2exmpls_lam50 (n=305) === f1_macro=0.4901 f1_binary=0.5770 acc=0.5049

=== prompt_d_mmr_classwise_wo_dict_4exmpls_lam50 (n=305) === f1_macro=0.5112 f1_binary=0.5689 acc=0.5180
[prompt_d_mmr_classwise_wo_dict_8exmpls_lam50] ВНИМАНИЕ: пропущено id из truth: 1, лишних id в pred: 0
[prompt_d_mmr_classwise_wo_dict_8exmpls_lam50] предупреждение: смержилось 304 из 305 строк

=== prompt_d_mmr_classwise_wo_dict_8exmpls_lam50 (n

In [21]:
def show_logs(logger_name="main", tail=50):
    log = logging.getLogger(logger_name)
    if not hasattr(log, "buffer"): print("Буфер не найден"); return
    for line in log.buffer.getvalue().splitlines()[-tail:]:
        print(line)
show_logs("main", 30); show_logs("parse", 100)


2026-06-18 10:41:14 | INFO     | main | [prompt_d_mmr_flat_with_dict_8exmpls_lam90] ТОКЕНЫ | ок=305/305 | prompt(sum=3591963, avg=11776.9) completion(sum=5087, avg=16.7) total(sum=3597050, avg=11793.6) | time=42.0s
2026-06-18 10:41:14 | INFO     | main | Сохранено 305 результатов → results_prompt_d_mmr_flat_with_dict_8exmpls_lam90.json
2026-06-18 10:41:14 | INFO     | main | ==================================================
2026-06-18 10:41:14 | INFO     | main | Вариант: prompt_d_mmr_flat_with_dict_16exmpls_lam90 | записей: 305
2026-06-18 10:41:54 | INFO     | main | [prompt_d_mmr_flat_with_dict_16exmpls_lam90] ТОКЕНЫ | ок=305/305 | prompt(sum=3840016, avg=12590.2) completion(sum=4623, avg=15.2) total(sum=3844639, avg=12605.4) | time=40.0s
2026-06-18 10:41:54 | INFO     | main | Сохранено 305 результатов → results_prompt_d_mmr_flat_with_dict_16exmpls_lam90.json
2026-06-18 10:41:54 | INFO     | main | ==================================================
2026-06-18 10:41:54 | INFO     | 

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image
)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import re, os

# ---------- 1. Шрифт с поддержкой кириллицы ----------
FONT_PATHS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "C:/Windows/Fonts/DejaVuSans.ttf",
    "/Library/Fonts/DejaVuSans.ttf",
]
FONT_NAME = "DejaVu"
for p in FONT_PATHS:
    if os.path.exists(p):
        pdfmetrics.registerFont(TTFont(FONT_NAME, p))
        break
else:
    FONT_NAME = "Helvetica"  # fallback (без кириллицы)

# ---------- 2. Данные ----------
data = """\
prompt_d_mmr_flat_with_dict_2exmpls_lam70,305,0.7869,0.6218,0.9417,0.7490,0.7819
prompt_d_mmr_flat_with_dict_2exmpls_lam90,305,0.7869,0.6218,0.9417,0.7490,0.7819
prompt_d_mmr_flat_with_dict_2exmpls_lam50,305,0.7869,0.6218,0.9417,0.7490,0.7819
prompt_d_mmr_flat_wo_dict_2exmpls_lam70,305,0.7705,0.6025,0.9417,0.7348,0.7663
prompt_d_mmr_flat_wo_dict_2exmpls_lam90,305,0.7705,0.6025,0.9417,0.7348,0.7663
prompt_d_mmr_flat_wo_dict_2exmpls_lam50,305,0.6951,0.5255,1.0000,0.6890,0.6950
prompt_d_mmr_classwise_with_dict_4exmpls_lam50,305,0.6590,0.4974,0.9417,0.6510,0.6588
prompt_d_mmr_classwise_with_dict_2exmpls_lam70,305,0.6492,0.4899,0.9417,0.6445,0.6491
prompt_d_mmr_classwise_with_dict_2exmpls_lam90,305,0.6426,0.4850,0.9417,0.6403,0.6426
prompt_d_mmr_classwise_with_dict_2exmpls_lam50,305,0.6393,0.4826,0.9417,0.6382,0.6393
prompt_d_mmr_classwise_with_dict_4exmpls_lam90,305,0.6393,0.4826,0.9417,0.6382,0.6393
prompt_d_mmr_classwise_with_dict_4exmpls_lam70,305,0.6328,0.4778,0.9417,0.6340,0.6328
prompt_d_mmr_flat_wo_dict_4exmpls_lam70,305,0.6262,0.4732,0.9417,0.6299,0.6262
prompt_d_mmr_flat_wo_dict_4exmpls_lam90,305,0.6230,0.4709,0.9417,0.6278,0.6229
prompt_d_mmr_flat_with_dict_4exmpls_lam90,305,0.6230,0.4709,0.9417,0.6278,0.6229
prompt_d_mmr_flat_with_dict_4exmpls_lam70,305,0.6197,0.4686,0.9417,0.6258,0.6196
prompt_d_mmr_flat_wo_dict_4exmpls_lam50,305,0.5738,0.4389,0.9417,0.5988,0.5721
prompt_d_mmr_flat_with_dict_4exmpls_lam50,305,0.5639,0.4330,0.9417,0.5933,0.5617
prompt_d_mmr_classwise_wo_dict_4exmpls_lam50,305,0.5180,0.4076,0.9417,0.5689,0.5112
prompt_d_mmr_classwise_wo_dict_4exmpls_lam90,305,0.5180,0.4076,0.9417,0.5689,0.5112
prompt_d_mmr_classwise_wo_dict_4exmpls_lam70,305,0.5148,0.4059,0.9417,0.5673,0.5075
prompt_d_mmr_classwise_wo_dict_32exmpls_lam70,303,0.5116,0.4104,1.0000,0.5819,0.4973
prompt_d_mmr_classwise_wo_dict_8exmpls_lam50,304,0.5099,0.4087,1.0000,0.5803,0.4957
prompt_d_mmr_classwise_wo_dict_32exmpls_lam90,301,0.5083,0.4056,1.0000,0.5771,0.4949
prompt_d_mmr_classwise_with_dict_16exmpls_lam90,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_wo_dict_16exmpls_lam70,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_32exmpls_lam70,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_32exmpls_lam50,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_16exmpls_lam70,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_8exmpls_lam70,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_wo_dict_8exmpls_lam70,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_8exmpls_lam50,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_8exmpls_lam90,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_wo_dict_16exmpls_lam90,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_with_dict_32exmpls_lam90,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_wo_dict_8exmpls_lam90,305,0.5082,0.4071,1.0000,0.5787,0.4941
prompt_d_mmr_classwise_wo_dict_2exmpls_lam50,305,0.5049,0.4055,1.0000,0.5770,0.4901
prompt_d_mmr_classwise_wo_dict_2exmpls_lam70,305,0.5049,0.4055,1.0000,0.5770,0.4901
prompt_d_mmr_classwise_wo_dict_2exmpls_lam90,305,0.5049,0.4055,1.0000,0.5770,0.4901
prompt_d_mmr_classwise_wo_dict_32exmpls_lam50,301,0.5050,0.4064,1.0000,0.5779,0.4898
prompt_d_mmr_classwise_with_dict_16exmpls_lam50,305,0.4918,0.3992,1.0000,0.5706,0.4741
prompt_d_mmr_classwise_wo_dict_16exmpls_lam50,302,0.4702,0.3916,1.0000,0.5628,0.4453
prompt_d_mmr_flat_with_dict_16exmpls_lam90,305,0.4525,0.3815,1.0000,0.5523,0.4238
prompt_d_mmr_flat_with_dict_32exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_16exmpls_lam70,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_8exmpls_lam70,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_16exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_32exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_8exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_8exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_16exmpls_lam50,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_8exmpls_lam70,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_32exmpls_lam90,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_32exmpls_lam70,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_32exmpls_lam70,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_8exmpls_lam90,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_16exmpls_lam90,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_8exmpls_lam90,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_wo_dict_32exmpls_lam90,305,0.4492,0.3801,1.0000,0.5508,0.4195
prompt_d_mmr_flat_with_dict_16exmpls_lam70,304,0.4474,0.3778,1.0000,0.5484,0.4183
"""

rows = [r.split(",") for r in data.strip().splitlines()]
df = pd.DataFrame(rows, columns=["version","n","accuracy","precision","recall","f1_binary","f1_macro"])
for c in ["n"]:
    df[c] = df[c].astype(int)
for c in ["accuracy","precision","recall","f1_binary","f1_macro"]:
    df[c] = df[c].astype(float)

# ---------- 3. Парсинг конфигурации из названия ----------
def parse(v):
    selector = "classwise" if "classwise" in v else "flat"
    dict_   = "with_dict" if "with_dict" in v else "wo_dict"
    m_ex    = re.search(r"_(\d+)exmpls", v)
    m_lam   = re.search(r"_lam(\d+)", v)
    return pd.Series({
        "selector": selector,
        "dict":     dict_,
        "n_examples": int(m_ex.group(1)) if m_ex else None,
        "lambda":  int(m_lam.group(1)) if m_lam else None,
    })

cfg = df["version"].apply(parse)
df = pd.concat([df, cfg], axis=1)
df_sorted = df.sort_values("f1_macro", ascending=False).reset_index(drop=True)

# ---------- 4. Графики ----------
os.makedirs("figs", exist_ok=True)

# Top-15 по f1_macro
plt.figure(figsize=(9,6))
top = df_sorted.head(15).iloc[::-1]
plt.barh(top["version"], top["f1_macro"], color="steelblue")
plt.xlabel("F1 macro")
plt.title("Top-15 версий по F1 macro")
plt.tight_layout()
plt.savefig("figs/top15.png", dpi=150)
plt.close()

# F1 vs n_examples по группам
plt.figure(figsize=(8,5))
for (sel, d), g in df.groupby(["selector","dict"]):
    grp = g.groupby("n_examples")["f1_macro"].mean().reset_index()
    plt.plot(grp["n_examples"], grp["f1_macro"], marker="o", label=f"{sel} / {d}")
plt.xlabel("Количество примеров (n_examples)")
plt.ylabel("F1 macro (среднее)")
plt.title("Зависимость F1 macro от числа примеров")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.savefig("figs/f1_vs_examples.png", dpi=150)
plt.close()

# ---------- 5. PDF ----------
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="RU",  fontName=FONT_NAME, fontSize=10, leading=14))
styles.add(ParagraphStyle(name="H1RU", fontName=FONT_NAME, fontSize=18, leading=22, spaceAfter=10, textColor=colors.HexColor("#1f3a68")))
styles.add(ParagraphStyle(name="H2RU", fontName=FONT_NAME, fontSize=13, leading=18, spaceBefore=10, spaceAfter=6, textColor=colors.HexColor("#1f3a68")))

doc = SimpleDocTemplate(
    "report.pdf",
    pagesize=landscape(A4),
    leftMargin=1.2*cm, rightMargin=1.2*cm,
    topMargin=1.2*cm, bottomMargin=1.2*cm,
)
story = []

# Заголовок
story.append(Paragraph("Отчёт по экспериментам с промптами (test)", styles["H1RU"]))
story.append(Paragraph(
    "Сводные метрики для семейства конфигураций <b>prompt_d_mmr_*</b>. "
    "Версии отсортированы по убыванию <b>F1 macro</b>.",
    styles["RU"]))
story.append(Spacer(1, 0.3*cm))

# Краткая статистика
best  = df_sorted.iloc[0]
worst = df_sorted.iloc[-1]
summary = f"""
<b>Всего версий:</b> {len(df)}<br/>
<b>Лучшая версия:</b> {best['version']} — F1 macro = {best['f1_macro']:.4f}, Accuracy = {best['accuracy']:.4f}<br/>
<b>Худшая версия:</b> {worst['version']} — F1 macro = {worst['f1_macro']:.4f}<br/>
<b>Средний F1 macro:</b> {df['f1_macro'].mean():.4f} &nbsp;|&nbsp;
<b>Медиана F1 macro:</b> {df['f1_macro'].median():.4f}
"""
story.append(Paragraph(summary, styles["RU"]))
story.append(Spacer(1, 0.4*cm))

# Ключевые выводы
story.append(Paragraph("Ключевые наблюдения", styles["H2RU"]))
bullets = [
    "Лучшие результаты достигаются при <b>малом числе примеров</b> (2–4) — больше примеров приводит к деградации метрик.",
    "Конфигурация <b>flat + with_dict + 2 примера</b> даёт лучший F1 macro = 0.7819 (независимо от λ).",
    "При <b>n_examples ≥ 8</b> у flat-вариантов метрики «схлопываются» к одинаковым значениям (≈0.42 F1 macro, recall=1.0) — модель, по-видимому, предсказывает один класс.",
    "Параметр <b>λ (MMR)</b> практически не влияет на качество в большинстве случаев.",
    "<b>classwise</b>-селектор стабильнее на больших n_examples, но в топе всё равно проигрывает <b>flat</b>.",
    "Наличие <b>словаря (with_dict)</b> даёт небольшой положительный эффект для flat-вариантов.",
]
for b in bullets:
    story.append(Paragraph(f"• {b}", styles["RU"]))
story.append(Spacer(1, 0.3*cm))

# Графики
story.append(Paragraph("Топ-15 версий по F1 macro", styles["H2RU"]))
story.append(Image("figs/top15.png", width=23*cm, height=15*cm))
story.append(PageBreak())

story.append(Paragraph("F1 macro в зависимости от числа примеров", styles["H2RU"]))
story.append(Image("figs/f1_vs_examples.png", width=20*cm, height=12*cm))
story.append(PageBreak())

# Таблица
story.append(Paragraph("Полная сводная таблица", styles["H2RU"]))

header = ["#", "Version", "n", "Accuracy", "Precision", "Recall", "F1 binary", "F1 macro"]
table_data = [header]
for i, row in df_sorted.iterrows():
    table_data.append([
        i+1,
        row["version"],
        row["n"],
        f"{row['accuracy']:.4f}",
        f"{row['precision']:.4f}",
        f"{row['recall']:.4f}",
        f"{row['f1_binary']:.4f}",
        f"{row['f1_macro']:.4f}",
    ])

col_widths = [0.9*cm, 10*cm, 1.2*cm, 2*cm, 2.2*cm, 1.8*cm, 2.2*cm, 2.2*cm]
tbl = Table(table_data, colWidths=col_widths, repeatRows=1)

style = TableStyle([
    ("FONT", (0,0), (-1,-1), FONT_NAME, 8),
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#1f3a68")),
    ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
    ("ALIGN", (0,0), (-1,-1), "CENTER"),
    ("ALIGN", (1,1), (1,-1), "LEFT"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
])
# подсветка топ-3
for i in range(1, min(4, len(table_data))):
    style.add("BACKGROUND", (0,i), (-1,i), colors.HexColor("#d6f5d6"))
tbl.setStyle(style)
story.append(tbl)

doc.build(story)
print("PDF создан: report.pdf")


PDF создан: report.pdf
